# 커스텀 경로 설계: 베지어 곡선

웨이포인트를 지정하여 G1-연속 베지어 스플라인 경로를 생성하고, 경로 기하학과 시뮬레이션 결과를 확인합니다.

This notebook shows how to create custom paths using `BezierPath` with user-defined waypoints, and how to use the built-in factory methods (`circle`, `figure_eight`).

**Author:** Suwon Lee, Kookmin University

In [ ]:
# -*- coding: utf-8 -*-
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from vfg_pathfollowing import BezierPath, Simulator

## 1. 웨이포인트 기반 경로 생성

5개의 웨이포인트를 지정하여 S자형 경로를 생성합니다. `tension` 파라미터는 곡률의 완만함을 조절합니다 (0에 가까울수록 꺾이고, 0.5에 가까울수록 완만).

In [ ]:
# 웨이포인트 정의 (S자형 경로)
waypoints = [(0, 0), (3, 1), (6, 0), (9, 2), (12, 0)]

# 베지어 경로 생성 (tension=0.33, Catmull-Rom 기본값)
path = BezierPath(waypoints, tension=0.33)
print(f"경로 총 길이: {path.total_length:.2f} m")

## 2. 경로 기하학 시각화

경로의 위치, 방위각, 곡률 프로파일을 확인합니다.

In [ ]:
# 경로를 따라 위치, 곡률 계산
s_arr = np.linspace(0, path.total_length, 500)
positions = np.array([path.position(s) for s in s_arr])
curvatures = np.array([path.curvature(s) for s in s_arr])
headings = np.array([path.heading(s) for s in s_arr])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 경로 형상
ax = axes[0]
ax.plot(positions[:, 0], positions[:, 1], 'b-', linewidth=2)
wp = np.array(waypoints)
ax.plot(wp[:, 0], wp[:, 1], 'ro', markersize=8, label='Waypoints')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_aspect('equal')
ax.set_title('Path Geometry')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 방위각
ax = axes[1]
ax.plot(s_arr, np.degrees(headings), 'g-', linewidth=1.5)
ax.set_xlabel('Arc length s [m]')
ax.set_ylabel('Heading [deg]')
ax.set_title('Heading Profile')
ax.grid(True, alpha=0.3)

# 곡률
ax = axes[2]
ax.plot(s_arr, curvatures, 'r-', linewidth=1.5)
ax.set_xlabel('Arc length s [m]')
ax.set_ylabel('Curvature [1/m]')
ax.set_title('Curvature Profile')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 3. 커스텀 경로에서 시뮬레이션

생성된 베지어 경로 위에서 LPV H-inf 제어기로 경로 추종 시뮬레이션을 실행합니다.

In [ ]:
# 시뮬레이션 실행
sim = Simulator(path, controller='lpv-hinf', speed=1.0)
result = sim.run(T=15.0)
result.plot(path=path)
plt.show()

## 4. 팩토리 메서드: 원형 및 8자 경로

`BezierPath`는 자주 사용되는 경로 형태를 팩토리 메서드로 제공합니다:
- `BezierPath.circle(radius)`: 원형 경로
- `BezierPath.figure_eight(radius)`: 8자형 경로

In [ ]:
# 원형 경로
circle_path = BezierPath.circle(radius=3.0)

# 8자형 경로
fig8_path = BezierPath.figure_eight(radius=2.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# 원형 경로 시각화
s_arr = np.linspace(0, circle_path.total_length, 500)
pos = np.array([circle_path.position(s) for s in s_arr])
axes[0].plot(pos[:, 0], pos[:, 1], 'b-', linewidth=2)
axes[0].set_aspect('equal')
axes[0].set_title('Circle (R=3.0m)')
axes[0].grid(True, alpha=0.3)

# 8자 경로 시각화
s_arr = np.linspace(0, fig8_path.total_length, 500)
pos = np.array([fig8_path.position(s) for s in s_arr])
axes[1].plot(pos[:, 0], pos[:, 1], 'r-', linewidth=2)
axes[1].set_aspect('equal')
axes[1].set_title('Figure-Eight (R=2.0m)')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()